In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

plt.rcParams['ps.useafm'] = True
plt.rcParams['pdf.use14corefonts'] = True
plt.rcParams['text.usetex'] = True
plt.rcParams["font.family"] = 'sans-serif'
plt.rcParams["font.sans-serif"] = ["Helvetica"] #["Avant Garde", "Computer Modern Serif","Helvetica"]
#plt.rcParams["text.latex.preamble"]= r"\usepackage{cmbright}"
plt.rcParams["text.latex.preamble"]= r"\usepackage[cm]{sfmath}"

In [23]:
%%bash
ls sim.log

sim.log


In [24]:
df_all=pd.read_csv("sim.log", delimiter=r'\s+')
df_all

,W#,Algorithm,BenchID,Name,Property,Value
0,W1,opt-unf_gmeanipc-min,0,OVERALL,STP,7.265
1,W1,opt-unf_gmeanipc-min,0,OVERALL,ANTT,1.10723
2,W1,opt-unf_gmeanipc-min,0,OVERALL,Unfairness,1.2113
3,W1,opt-unf_gmeanipc-min,0,OVERALL,UnfairnessCoV,0.07474
4,W1,opt-unf_gmeanipc-min,0,OVERALL,JainFairness,0.99445
...,...,...,...,...,...,...
41511,W99,opt-unf_stp-min,16,namd06,way_count,6
41512,W99,opt-unf_stp-min,16,namd06,way_mask,0x3f
41513,W99,opt-unf_stp-min,16,namd06,slowdown,1.00288
41514,W99,opt-unf_stp-min,16,namd06,ipc,2.10511


In [25]:
df_all["W#"].unique()

<StringArray>
[  'W1',  'W10', 'W100', 'W101', 'W102', 'W103', 'W104', 'W105', 'W106',
 'W107',
 ...
  'W90',  'W91',  'W92',  'W93',  'W94',  'W95',  'W96',  'W97',  'W98',
  'W99']
Length: 127, dtype: str

In [26]:
df_all.head(50)

,W#,Algorithm,BenchID,Name,Property,Value
0,W1,opt-unf_gmeanipc-min,0,OVERALL,STP,7.265
1,W1,opt-unf_gmeanipc-min,0,OVERALL,ANTT,1.10723
2,W1,opt-unf_gmeanipc-min,0,OVERALL,Unfairness,1.2113
3,W1,opt-unf_gmeanipc-min,0,OVERALL,UnfairnessCoV,0.07474
4,W1,opt-unf_gmeanipc-min,0,OVERALL,JainFairness,0.99445
5,W1,opt-unf_gmeanipc-min,0,OVERALL,GmeanSpeedup,0.906
6,W1,opt-unf_gmeanipc-min,0,OVERALL,GmeanIPC,1.793
7,W1,opt-unf_gmeanipc-min,0,OVERALL,AggregateIPC,15.41763
8,W1,opt-unf_gmeanipc-min,0,OVERALL,HmeanSpeedup,0.90316
9,W1,opt-unf_gmeanipc-min,0,OVERALL,M1,5.847


In [27]:
df_all["Property"].unique()

<StringArray>
[          'STP',          'ANTT',    'Unfairness', 'UnfairnessCoV',
  'JainFairness',  'GmeanSpeedup',      'GmeanIPC',  'AggregateIPC',
  'HmeanSpeedup',            'M1', 'explored_sols',    'cluster_id',
     'way_count',      'way_mask',      'slowdown',           'ipc',
     'ipc_alone']
Length: 17, dtype: str

In [28]:
df_all["Algorithm"].unique()

<StringArray>
['opt-unf_gmeanipc-min',     'opt-unf_gmsp-min',     'opt-unf_hmsp-min',
      'opt-unf_stp-min']
Length: 4, dtype: str

In [29]:
workloads=df_all["W#"].unique()
schemes=df_all["Algorithm"].unique()
metrics=["Unfairness","STP","HmeanSpeedup","GmeanSpeedup","GmeanIPC"]

rows=[]
for w in workloads:
    for s in schemes:
        sdf=df_all[(df_all["Algorithm"]==s)&(df_all["W#"]==w)]
        row=[w,s]
        for m in metrics:
            #print (w,s,m)
            v=int(float(sdf[sdf.Property==m].Value.values[0])*10000)
            row.append(v)
        rows.append(row)
        

sel=pd.DataFrame(rows,columns=["Workload","Scheme"]+metrics)            
sel
        



,Workload,Scheme,Unfairness,STP,HmeanSpeedup,GmeanSpeedup,GmeanIPC
0,W1,opt-unf_gmeanipc-min,12113,72650,9031,9060,17930
1,W1,opt-unf_gmsp-min,12113,72650,9031,9060,17930
2,W1,opt-unf_hmsp-min,12113,72650,9031,9060,17930
3,W1,opt-unf_stp-min,12113,72650,9031,9060,17930
4,W10,opt-unf_gmeanipc-min,12602,71930,8917,8950,14310
...,...,...,...,...,...,...,...
503,W98,opt-unf_stp-min,12430,149130,9249,9290,16350
504,W99,opt-unf_gmeanipc-min,12979,147510,9113,9170,15620
505,W99,opt-unf_gmsp-min,13062,147480,9110,9170,15610
506,W99,opt-unf_hmsp-min,13062,147480,9110,9170,15610


## Try to find if some experiments have different metrics

In [30]:
groups = ["Workload", "Scheme"]

# Count distinct values for each metric within each group
unique_counts = sel.groupby(groups)[metrics].nunique(dropna=False)

# Verify all metrics are identical within every group
assert unique_counts.eq(1).all().all(), "Some experiments have different metrics!"

## Identify groups with mismatches (None is shown)

In [31]:
mismatches = unique_counts[unique_counts.ne(1).any(axis=1)]

print(mismatches)

Empty DataFrame
Columns: [Unfairness, STP, HmeanSpeedup, GmeanSpeedup, GmeanIPC]
Index: []
